# Pratica — Lezione 8. Gradienti di politica e metodi Actor-Critic

**DIH DATalife — Corso di Reinforcement Learning (RL) da zero**

In questa pratica programmi tu stesso il cuore di REINFORCE: il calcolo dei ritorni scontati e la perdita del gradiente di politica. Poi aggiungi una baseline e verifichi in prima persona quanto aiuta a ridurre la varianza dell'addestramento.

Usi lo stesso ambiente della lezione: CartPole-v1, dove un carrello deve mantenere in equilibrio un'asta muovendosi a sinistra o a destra.

## Preparazione

**Esegui le celle seguenti in ordine.** Installano gymnasium, importano le librerie e ti lasciano pronti due elementi già completi e funzionanti: la rete della politica e la raccolta di un episodio. Non serve modificarli.

In [ ]:
# Instala gymnasium (PyTorch ya viene preinstalado en Colab)
!pip install gymnasium

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

### La rete della politica (già completa)

`RedPolitica` rappresenta `π_θ(a|s)`: riceve lo stato di CartPole e restituisce una distribuzione di probabilità sulle sue due azioni, spingere a sinistra o a destra. È la stessa rete vista in classe.

In [ ]:
class RedPolitica(nn.Module):
    """Red neuronal pequeña que representa la política π_θ(a|s). Devuelve una
    distribución de probabilidad sobre las dos acciones posibles."""

    def __init__(self, n_entradas, n_acciones, n_ocultas=128):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(n_entradas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, n_acciones),
        )

    def forward(self, estado):
        logits = self.red(estado)
        return torch.softmax(logits, dim=-1)

### Raccogliere un episodio (già completo)

`ejecutar_episodio` segue la politica attuale dal reset dell'ambiente fino alla fine dell'episodio, e salva la log-probabilità di ogni azione intrapresa (ti serve per il gradiente di politica) insieme alla ricompensa di ogni passo. È la stessa funzione della lezione.

In [ ]:
def ejecutar_episodio(env, politica):
    """Recolecta un episodio completo siguiendo la política actual. Devuelve
    la log-probabilidad de cada acción tomada y la recompensa de cada paso."""
    log_probs = []
    recompensas = []
    estado, _ = env.reset()
    terminado = False
    truncado = False

    while not (terminado or truncado):
        estado_tensor = torch.FloatTensor(estado).unsqueeze(0)
        probs = politica(estado_tensor)
        distribucion = torch.distributions.Categorical(probs)
        accion = distribucion.sample()

        log_probs.append(distribucion.log_prob(accion))
        estado, recompensa, terminado, truncado, _ = env.step(accion.item())
        recompensas.append(recompensa)

    return log_probs, recompensas

## Esercizio 1 — Completa i ritorni e la perdita di REINFORCE

In classe hai visto che REINFORCE richiede due ingredienti per episodio: il ritorno di ogni passo (la somma scontata delle ricompense future) e la perdita che spinge verso l'alto la probabilità delle azioni andate bene.

Qui sotto trovi `calcular_retornos` e `entrenar_reinforce` con due `# TODO`. Il resto del ciclo di addestramento —creare l'ambiente, la politica, l'ottimizzatore, eseguire gli episodi e aggiornare i parametri— è già costruito.

**Il tuo compito è completare i due `# TODO`:**

1. In `calcular_retornos`, dentro il ciclo `for t in reversed(...)`, accumula il ritorno scontato: `G = recompensas[t] + gamma * G`, riutilizzando il `G` già calcolato al passo successivo
2. In `entrenar_reinforce`, dentro il ciclo `for log_prob, G in zip(...)`, calcola il termine della perdita per quel passo: `-log_prob * G` (il segno negativo trasforma la salita del gradiente che vuoi nella discesa che usa PyTorch)

**Suggerimenti**

- La formula di `calcular_retornos` è esattamente quella vista in classe: `G_t = r_t + gamma * G_{t+1}`, calcolata dalla fine verso l'inizio
- La perdita completa è `-sum(log_prob(a_t|s_t) * G_t)`: qui ti manca solo il termine di un passo, la somma è già fatta con `torch.stack(...).sum()`
- Con i due `# TODO` non completati, la cella di prova qui sotto viene eseguita senza errori, ma la ricompensa media non migliora: resta intorno a 20-30, la ricompensa tipica di una politica che agisce a caso. Questo è il segnale che ti manca ancora qualcosa da completare

In [ ]:
def calcular_retornos(recompensas, gamma=0.99):
    """Calcula el retorno 'reward-to-go' en cada paso del episodio:
    G_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ...
    Se recorre la lista de atrás hacia adelante para reutilizar el cálculo
    del paso siguiente en cada paso anterior."""
    retornos = np.zeros(len(recompensas))
    G = 0.0
    for t in reversed(range(len(recompensas))):
        # TODO: acumula el retorno con descuento, reutilizando el G del paso
        # siguiente: G = recompensas[t] + gamma * G
        retornos[t] = G
    return retornos


def entrenar_reinforce(n_episodios, gamma=0.99, lr=1e-2, seed=0, verbose=True):
    """Bucle de entrenamiento de REINFORCE. Ya está completo: solo te falta
    terminar calcular_retornos (arriba) y el término de la pérdida (abajo)."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    env = gym.make("CartPole-v1")
    env.reset(seed=seed)  # sembramos el entorno una vez, para que el experimento sea reproducible
    n_entradas = env.observation_space.shape[0]
    n_acciones = env.action_space.n

    politica = RedPolitica(n_entradas, n_acciones)
    optimizador = optim.Adam(politica.parameters(), lr=lr)
    historial_recompensas = []

    for episodio in range(n_episodios):
        log_probs, recompensas = ejecutar_episodio(env, politica)
        retornos = calcular_retornos(recompensas, gamma)
        retornos_tensor = torch.FloatTensor(retornos)

        terminos_perdida = []
        for log_prob, G in zip(log_probs, retornos_tensor):
            # TODO: sustituye la siguiente línea por el término correcto de
            # la pérdida de REINFORCE para este paso: -log_prob * G
            termino = log_prob * 0.0
            terminos_perdida.append(termino)
        perdida = torch.stack(terminos_perdida).sum()

        optimizador.zero_grad()
        perdida.backward()
        optimizador.step()

        historial_recompensas.append(sum(recompensas))

        if verbose and (episodio + 1) % 50 == 0:
            media_reciente = np.mean(historial_recompensas[-50:])
            print(f"Episodio {episodio + 1}: recompensa media (últimos 50) = {media_reciente:.1f}")

    env.close()
    return historial_recompensas

**Esegui la cella seguente per provare la tua implementazione.** Confronta la ricompensa media dei primi 20 episodi con quella degli ultimi 20: se il tuo codice funziona, la seconda dovrebbe essere parecchie volte più alta della prima.

In [ ]:
historial_prueba = entrenar_reinforce(n_episodios=250, seed=0)

print("\nRecompensa media, primeros 20 episodios:", np.mean(historial_prueba[:20]))
print("Recompensa media, últimos 20 episodios:", np.mean(historial_prueba[-20:]))

<details>
<summary>💡 Vedi la soluzione</summary>

```python
def calcular_retornos(recompensas, gamma=0.99):
    """Calcola il ritorno 'reward-to-go' a ogni passo dell'episodio:
    G_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ...
    Si percorre la lista dalla fine verso l'inizio per riutilizzare il calcolo
    del passo successivo in ogni passo precedente."""
    retornos = np.zeros(len(recompensas))
    G = 0.0
    for t in reversed(range(len(recompensas))):
        # Accumuliamo il ritorno scontato, riutilizzando il G del passo successivo
        G = recompensas[t] + gamma * G
        retornos[t] = G
    return retornos


def entrenar_reinforce(n_episodios, gamma=0.99, lr=1e-2, seed=0, verbose=True):
    torch.manual_seed(seed)
    np.random.seed(seed)
    env = gym.make("CartPole-v1")
    env.reset(seed=seed)
    n_entradas = env.observation_space.shape[0]
    n_acciones = env.action_space.n

    politica = RedPolitica(n_entradas, n_acciones)
    optimizador = optim.Adam(politica.parameters(), lr=lr)
    historial_recompensas = []

    for episodio in range(n_episodios):
        log_probs, recompensas = ejecutar_episodio(env, politica)
        retornos = calcular_retornos(recompensas, gamma)
        retornos_tensor = torch.FloatTensor(retornos)

        terminos_perdida = []
        for log_prob, G in zip(log_probs, retornos_tensor):
            # El termino de la perdida de REINFORCE para este paso
            termino = -log_prob * G
            terminos_perdida.append(termino)
        perdida = torch.stack(terminos_perdida).sum()

        optimizador.zero_grad()
        perdida.backward()
        optimizador.step()

        historial_recompensas.append(sum(recompensas))

        if verbose and (episodio + 1) % 50 == 0:
            media_reciente = np.mean(historial_recompensas[-50:])
            print(f"Episodio {episodio + 1}: recompensa media (ultimi 50) = {media_reciente:.1f}")

    env.close()
    return historial_recompensas
```
</details>

## Esercizio 2 — Aggiungi una baseline e confronta la stabilità

Ora hai REINFORCE completo. In classe hai visto che sottrarre una baseline al ritorno riduce la varianza senza introdurre bias. La baseline più semplice che puoi costruire senza addestrare una rete aggiuntiva è la media degli stessi ritorni dell'episodio: `A_t = G_t - media(G)`.

**Perché tu possa andare avanti anche se la tua soluzione dell'esercizio uno non è ancora pronta**, la cella seguente include `entrenar_con_linea_base`, una copia già completa e funzionante di REINFORCE (con il proprio calcolo dei ritorni, indipendente dall'esercizio uno), con un parametro `usar_linea_base`.

**Il tuo compito è completare l'unico `# TODO`:** quando `usar_linea_base` è vero, calcola il vantaggio sottraendo a `retornos` la sua stessa media.

**Suggerimenti**

- NumPy calcola la media di un array con `.mean()`: `retornos.mean()`
- Il vantaggio ha la stessa forma di `retornos`: stai sottraendo solo un numero (la media), quindi la sottrazione si applica a ogni elemento in una volta
- Quando `usar_linea_base` è falso, `ventaja` deve restare uguale a `retornos`, senza alcuna modifica

In [ ]:
def _retornos_descontados(recompensas, gamma):
    """Calculo de los retornos, identico al del ejercicio uno pero ya
    completo, para que este ejercicio no dependa de tu solucion anterior."""
    retornos = np.zeros(len(recompensas))
    G = 0.0
    for t in reversed(range(len(recompensas))):
        G = recompensas[t] + gamma * G
        retornos[t] = G
    return retornos


def entrenar_con_linea_base(n_episodios, gamma=0.99, usar_linea_base=False, lr=1e-2, seed=0, verbose=True):
    """Copia completa de REINFORCE. Con usar_linea_base=False es idéntico al
    REINFORCE del ejercicio uno. Con usar_linea_base=True, resta la media de
    los retornos del episodio antes de calcular la pérdida."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    env = gym.make("CartPole-v1")
    env.reset(seed=seed)
    n_entradas = env.observation_space.shape[0]
    n_acciones = env.action_space.n

    politica = RedPolitica(n_entradas, n_acciones)
    optimizador = optim.Adam(politica.parameters(), lr=lr)
    historial_recompensas = []

    for episodio in range(n_episodios):
        log_probs, recompensas = ejecutar_episodio(env, politica)
        retornos = _retornos_descontados(recompensas, gamma)

        if usar_linea_base:
            # TODO: calcula la ventaja restando a `retornos` su propia media
            ventaja = retornos
        else:
            ventaja = retornos

        ventaja_tensor = torch.FloatTensor(ventaja)
        perdida = torch.stack(
            [-log_prob * A for log_prob, A in zip(log_probs, ventaja_tensor)]
        ).sum()

        optimizador.zero_grad()
        perdida.backward()
        optimizador.step()

        historial_recompensas.append(sum(recompensas))

        if verbose and (episodio + 1) % 50 == 0:
            media_reciente = np.mean(historial_recompensas[-50:])
            print(f"Episodio {episodio + 1}: recompensa media (últimos 50) = {media_reciente:.1f}")

    env.close()
    return historial_recompensas

**Esegui la cella seguente per addestrare due agenti con lo stesso seed, uno senza baseline e uno con essa, e tracciare le due curve.** Calcoli anche un numero che riassume quanto "saltano" i valori tra episodi consecutivi: la media del valore assoluto della differenza tra ogni episodio e il precedente. Più è basso, più stabile è l'addestramento.

In [ ]:
N_EPISODIOS = 300
SEED = 0

print("--- Entrenando sin línea base ---")
historial_sin = entrenar_con_linea_base(N_EPISODIOS, usar_linea_base=False, seed=SEED, verbose=False)

print("--- Entrenando con línea base ---")
historial_con = entrenar_con_linea_base(N_EPISODIOS, usar_linea_base=True, seed=SEED, verbose=False)


def volatilidad(historial):
    """Media del salto absoluto entre episodios consecutivos: cuanto más
    bajo, menos brusco es el entrenamiento episodio a episodio."""
    return np.mean(np.abs(np.diff(historial)))


print(f"\nVolatilidad sin línea base: {volatilidad(historial_sin):.1f}")
print(f"Volatilidad con línea base: {volatilidad(historial_con):.1f}")

plt.figure(figsize=(10, 5))
plt.plot(historial_sin, alpha=0.6, label="Sin línea base")
plt.plot(historial_con, alpha=0.6, label="Con línea base")
plt.xlabel("Episodio")
plt.ylabel("Recompensa total")
plt.title("REINFORCE en CartPole-v1: con y sin línea base")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Attenzione alla lettura del risultato**: la baseline riduce la varianza del *gradiente* in modo dimostrabile, in media e con molti campioni. Ma un singolo addestramento di poche centinaia di episodi resta un campione piccolo, quindi non sempre vedrai a colpo d'occhio una curva più liscia: in alcune esecuzioni aiuta chiaramente, in altre si nota appena, e anche questa è una lezione reale sul RL, non un errore del tuo codice. Se vuoi un confronto più solido, prova con due o tre seed diversi (cambia `SEED`) e guarda la tendenza generale, più che una singola esecuzione. Discuti con il tuo team quali curve ti sembrano più stabili e perché.

**Prova anche valori diversi di `gamma`** (per esempio, 0,9 contro 0,99, dentro `entrenar_con_linea_base`) e annota come cambia la velocità di apprendimento.

<details>
<summary>💡 Vedi la soluzione</summary>

```python
def _retornos_descontados(recompensas, gamma):
    retornos = np.zeros(len(recompensas))
    G = 0.0
    for t in reversed(range(len(recompensas))):
        G = recompensas[t] + gamma * G
        retornos[t] = G
    return retornos


def entrenar_con_linea_base(n_episodios, gamma=0.99, usar_linea_base=False, lr=1e-2, seed=0, verbose=True):
    torch.manual_seed(seed)
    np.random.seed(seed)
    env = gym.make("CartPole-v1")
    env.reset(seed=seed)
    n_entradas = env.observation_space.shape[0]
    n_acciones = env.action_space.n

    politica = RedPolitica(n_entradas, n_acciones)
    optimizador = optim.Adam(politica.parameters(), lr=lr)
    historial_recompensas = []

    for episodio in range(n_episodios):
        log_probs, recompensas = ejecutar_episodio(env, politica)
        retornos = _retornos_descontados(recompensas, gamma)

        if usar_linea_base:
            # La baseline è la media degli stessi ritorni dell'episodio
            ventaja = retornos - retornos.mean()
        else:
            ventaja = retornos

        ventaja_tensor = torch.FloatTensor(ventaja)
        perdida = torch.stack(
            [-log_prob * A for log_prob, A in zip(log_probs, ventaja_tensor)]
        ).sum()

        optimizador.zero_grad()
        perdida.backward()
        optimizador.step()

        historial_recompensas.append(sum(recompensas))

        if verbose and (episodio + 1) % 50 == 0:
            media_reciente = np.mean(historial_recompensas[-50:])
            print(f"Episodio {episodio + 1}: recompensa media (ultimi 50) = {media_reciente:.1f}")

    env.close()
    return historial_recompensas
```
</details>

## Per continuare a esercitarti

**Confronta la volatilità mediando più seed**: ripeti l'addestramento con e senza baseline usando tre o quattro valori diversi di `SEED`, e calcola la media della volatilità di ciascun gruppo. Il vantaggio della baseline si mantiene quando mediate più esecuzioni?

Con questo completi la lezione otto. Nella lezione 9 passi alla pianificazione con modelli: Dyna-Q e la ricerca ad albero di Monte Carlo (Monte Carlo Tree Search).